# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amal-ali-u/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I chose Ranking because my goal is to prioritize content pages for refresh. The output will be an ordered list of pages, from highest to lowest priority, based on content and search signals. This will help the content and SEO teams decide which pages to investigate and update first.
*Classification, clustering, ranking, or scoring — which one, and why?*

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy
*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My goal is to prioritize content pages whose search performance is declining.the dataset contains `trend_direction`, which identifies observed trend categories, including `down`, `stable`, and `up`. I can use this field to identify pages with observed declines for an initial rule-based prioritization baseline.

However, the dataset does not contain outcomes showing whether a page improved after being refreshed. Therefore, observed decline is a proxy for intervention priority, not a label for successful refresh.

I will not use `trend_direction` or `trend_pct` as predictive features because they represent or contribute to the observed trend outcome. I will investigate other available signals to build and evaluate a useful prioritization approach.


In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

For an initial ranking baseline, I will use Precision@K to measure the proportion of pages in the top K recommendations that have an observed `down` trend.

This metric evaluates whether the ranking prioritizes pages with recorded performance declines. It does not measure whether refreshing those pages will improve performance, because post-refresh outcomes are not available in the current dataset.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
The unit of analysis is one content page. Each row represents a unique content item, identified by content_id.

The dataset contains 30,000 rows and 30,000 unique content IDs. Each row includes content characteristics, search-performance metrics, and an observed trend category.

The content_id identifies the page, while client_id supports grouping pages by client. These IDs are used for identification and grouping, not as ranking signals. The trend_direction column will be used to evaluate whether the ranking prioritizes pages with declining performance, not as a predictive feature.

In [44]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from pathlib import Path

data_path = Path(
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")
# Load the dataset if it has not been loaded already
if "df" not in globals():
    df = pd.read_csv(data_path)

# Select the columns relevant to page prioritization
unit_df = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "ctr",
        "avg_position",
        "impression_tier",
        "trend_direction"
    ]
].copy()

# Zero means missing position data
unit_df["avg_position"] = unit_df["avg_position"].replace(0, float("nan"))

print("Number of rows:", len(unit_df))
print("Number of unique content IDs:", unit_df["content_id"].nunique())

display(unit_df.head(10))


Number of rows: 30000
Number of unique content IDs: 30000


,content_id,client_id,content_type,main_intent,ctr,avg_position,impression_tier,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,0.76,10.6,good,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,0.05,20.3,good,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,0.09,36.5,good,down
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,0.49,6.2,good,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,0.13,44.0,good,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,0.03,8.5,good,down
6,content_9a34b442b552,client_8722616204,keyword article,informational,0.00,7.0,low,down
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,0.06,21.2,moderate,stable
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,0.09,46.0,excellent,down
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,0.16,4.9,moderate,down


In [45]:
unit_df = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "ctr",
        "avg_position",
        "impression_tier",
        "trend_direction"
    ]
].copy()

print("Number of rows:", len(unit_df))
print("Unique content IDs:", unit_df["content_id"].nunique())

display(unit_df.head(10))

Number of rows: 30000
Unique content IDs: 30000


,content_id,client_id,content_type,main_intent,ctr,avg_position,impression_tier,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,0.76,10.6,good,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,0.05,20.3,good,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,0.09,36.5,good,down
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,0.49,6.2,good,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,0.13,44.0,good,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,0.03,8.5,good,down
6,content_9a34b442b552,client_8722616204,keyword article,informational,0.00,7.0,low,down
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,0.06,21.2,moderate,stable
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,0.09,46.0,excellent,down
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,0.16,4.9,moderate,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule based on a single metric, such as low CTR, may overlook other important signals related to page performance. For example, a page may have a reasonable CTR but still experience declining impressions, or it may have a weak average search position.

A ranking approach that considers multiple content and search signals could help prioritize pages more effectively by capturing patterns that a single threshold might miss.

However, machine learning is not guaranteed to outperform a fixed rule. I would compare the ranking approach against a simple baseline using Precision@K, with trend_direction = down as the relevance label. Since post-refresh outcomes are unavailable, this evaluation would measure how well the ranking identifies currently declining pages, not whether refreshing them improves performance.

In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- **Task type:** Ranking.
- **Decision supported:** Prioritize content pages for investigation and potential refresh.
- **Target or proxy:** The observed `trend_direction = down` category is used as a relevance label for evaluating the ranking, not as a predictive feature.
- **Success metric:** Precision@K, measuring the proportion of declining pages among the top K ranked pages.
- **Unit of analysis:** One content page per row, identified by `content_id`.
- **Why ML:** Multiple content and search signals may capture patterns that a single fixed rule misses.
- **Main limitation:** The dataset does not contain post-refresh outcomes, so it cannot directly measure whether a refresh improves page performance.
- **Data leakage check:** Do not use `trend_direction` or `trend_pct` as predictive features. Do not treat `avg_position = 0` as a real search position.
- **Validation:** Compare any proposed ranking approach with a simple baseline before claiming that machine learning performs better.